# ArcFace

In [1]:
!pip3 install insightface onnxruntime opencv-python
import cv2, torch, insightface, os
import numpy as np
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image
from torch.nn.functional import cosine_similarity
print("Current working directory:", os.getcwd())

# Load ResNet100
app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=-1, det_size=(640, 640))

Current working directory: g:\.thesis\named-ai\data-preprocessing


c:\Users\julia\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [6]:
#get embeddings FIX ERROR HANDLING
def get_embeddings_resnet100(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable: {image_path}")

    faces = app.get(img)
    if len(faces) == 0:
        print(f"No face detected in {image_path}")
        return None
    print(f"Face detected in {image_path}")
    emb = torch.tensor(faces[0].embedding).squeeze()
    emb = torch.nn.functional.normalize(emb, p=2, dim=0)
    return emb

In [ ]:
#facebank method (substitute for KNN)
def build_facebank(base_dir):
    facebank = {}
    for person_name in os.listdir(base_dir):
        person_dir = os.path.join(base_dir, person_name)
        if not os.path.isdir(person_dir):
            continue

        embeddings = []

        for img_name in os.listdir(base_dir):
            img_path = os.path.join(person_dir, img_name)
            emb = get_embeddings_resnet100(img_path)
            if emb is not None:
                embeddings.append(emb)
        if embeddings:
            facebank[person_name] = embeddings

    return facebank

In [ ]:
#not needed anymore once facebank is implemented
def average_embeddings(image_paths):
    embeddings = []
    for path in image_paths:
        emb = get_embeddings_resnet100(path)
        if emb is not None:
            embeddings.append(emb)

    if not embeddings:
        return None  # no valid embeddings

    avg_embedding = torch.stack(embeddings).mean(dim=0)
    # Normalize again after averaging
    avg_embedding = torch.nn.functional.normalize(avg_embedding, p=2, dim=0)
    return avg_embedding

In [4]:
#recognize face FIX ERROR HANDLING
def recognize_face(test_embedding, face_db, threshold=0.6):
    if test_embedding is None:
        return "Unknown", 0.0

    max_sim = 0
    identity = "Unknown"

    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        sim_val = sim.item()

        if sim_val > max_sim and sim_val > threshold:
            max_sim = sim_val
            identity = name

    return identity, max_sim

In [12]:
%cd Face_Dataset/MobileFaceNet
print("Current working directory:", os.getcwd())
face_db_r100 = {}
#db_images = {}
#test_images = {}
    #"Akshay": ["orig-img/orig-img/akshay/Akshay Kumar_3.jpg", "orig-img/orig-img/akshay/Akshay Kumar_1.jpg", "orig-img/orig-img/akshay/Akshay Kumar_2.jpg"],
    #"Alexandra": ["orig-img/orig-img/alexandra/Alexandra Daddario_0.jpg", "orig-img/orig-img/alexandra/Alexandra Daddario_1.jpg", "orig-img/orig-img/alexandra/Alexandra Daddario_2.jpg"]
base_dir = "final-images"
face_db = {}
people = {
    "Alexandra Daddario": "Alexandra Daddario",
    "Andy Samberg": "Andy Samberg",
    "Brad Pitt": "Brad Pitt",
    "Camila Cabello": "Camila Cabello",
    "The Rock": "Dwayne Johnson",
    "Elizabeth Olsen": "Elizabeth Olsen",
    "Henry Cavill": "Henry Cavill",
    "Margot Robbie": "Margot Robbie",
    "Robert Downey Jr.": "Robert Downey Jr",   
    "Tom Cruise": "Tom Cruise"
}

for name, folder in people.items():
    folder_path = os.path.join(base_dir, folder)
    image_files = [
        os.path.join(folder_path, f"{folder}_{i}.jpg")
        for i in range(45)   # 0–45 inclusive
    ]
    #image_files = [os.path.join(folder_path, f)
    #               for f in os.listdir(folder_path)
     #              if f.lower().endswith((".jpg", ".jpeg", ".png"))
    #]

    db_embeddings = []
    for i in range(min(45,len(image_files))):
        emb = get_embeddings_resnet100(image_files[i])
        if emb is not None:
            db_embeddings.append(emb)
    if db_embeddings:
        avg_emb = torch.stack(db_embeddings).mean(dim=0)
        avg_emb = torch.nn.functional.normalize(avg_emb, p=2, dim=0)
        face_db[name] = avg_emb

    #db_images[name] = image_files[:45]
    #test_images[name] = image_files[45:]

#for name, paths in db_images.items():
#    avg_emb = average_embeddings(paths)
#    if avg_emb is not None:
#        face_db[name] = avg_emb
#for name, paths in known_faces.items():
#    if isinstance(paths, str):
#        paths = [paths]  # handle single string input
#    avg_emb = average_embeddings(paths)
#    if avg_emb is not None:
#        face_db_r100[name] = avg_emb
# build db
#face_db_r100 = {}
#face_db_r100["Kyle"] = get_embeddings_resnet100("my_images/kyle_183.jpg")
#face_db_r100["Tom Cruise"] = get_embeddings_resnet100("my_images/Tom Cruise_12.jpg")
#face_db_r100["Tom CruiseTest"] = get_embeddings_resnet100("my_images/testtom.jpg")
#print("Current working directory:", os.getcwd())
#face_db_r100["Kumar"] = get_embeddings_resnet100("orig-img/orig-img/akshay/Akshay Kumar_0.jpg")

[WinError 3] The system cannot find the path specified: 'Face_Dataset/MobileFaceNet'
g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet
Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_0.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_1.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_2.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_3.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_4.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_5.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_6.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_7.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_8.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_9.jpg
Face detected in

In [13]:
#test recognition
#test_embedding_r100 = get_embeddings_resnet100("my_images/testtom.jpg")
test_embedding_r100 = get_embeddings_resnet100("final-images/Elizabeth Olsen/Elizabeth Olsen_59.jpg")

print(f"{test_embedding_r100}")
identity_r100, confidence_r100 = recognize_face(test_embedding_r100, face_db, threshold=0.6)
print(f"[ResNet100] Identified as: {identity_r100} (Confidence: {confidence_r100:.2f})")


Face detected in final-images/Elizabeth Olsen/Elizabeth Olsen_59.jpg
tensor([ 3.6881e-02,  1.9824e-02,  2.8946e-02, -2.3265e-02, -6.0046e-02,
        -7.5545e-02, -2.4536e-02, -5.3242e-02,  1.0358e-02, -2.7836e-02,
         1.4786e-02, -1.1160e-02,  5.2027e-02, -6.2493e-03,  4.7888e-02,
         6.8601e-02,  5.8135e-02, -5.2031e-02,  1.9837e-02, -1.5451e-02,
         6.9293e-02,  8.3127e-02,  1.5711e-02,  2.1742e-02, -1.1120e-02,
        -2.4492e-02,  1.6516e-02,  8.2329e-03, -2.5619e-02,  1.6724e-02,
         6.2947e-02, -2.6189e-02,  2.7246e-02,  5.2105e-02, -9.2698e-03,
         2.8400e-02,  5.3919e-02, -2.5577e-03,  1.3660e-02, -2.8638e-02,
        -6.5804e-02,  3.1932e-02, -3.4854e-02, -3.5533e-03,  3.2375e-03,
        -1.6491e-02, -1.8948e-02, -5.2702e-02, -8.7155e-03,  1.3612e-02,
         6.0468e-02,  3.2448e-02,  2.0037e-02, -6.5725e-02,  4.8116e-02,
         5.3133e-02,  2.9529e-03,  1.4815e-02, -4.7138e-02, -5.3418e-02,
        -7.2724e-02, -9.7162e-02, -4.2905e-02, -5.8130e